In [14]:
import os, sys, importlib
os.chdir("C:\\Users\\seude\\Desktop\\DPforAgenticEnv\\tabularqa_dp")     
sys.path.insert(0, os.getcwd())
print(sys.version)                              # 3.12+ 이어야 함

import numpy as np, pandas as pd
import dp_agent_wrap as W
importlib.reload(W)

3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]


<module 'dp_agent_wrap' from 'C:\\Users\\seude\\Desktop\\DPforAgenticEnv\\tabularqa_dp\\dp_agent_wrap.py'>

In [21]:
DS_NAME  = "CardBase"
COL      = "Credit_Limit"
# QUESTION = "What is the average Credit_Limit?"
QUESTION = "What is the highest Credit_Limit?"
EPS      = 1.0

# 1) 로더가 로컬 parquet 을 실제로 잡는지 (HuggingFace 폴백 차단)
df   = W.assert_local_table(DS_NAME)
n    = len(df)
SENS = (df[COL].max() - df[COL].min()) / n     # 데이터 의존적. 공개 도메인 범위로 간주.
# TRUE = df[COL].mean()
TRUE = df[COL].max()
print(df.shape, "SENS =", SENS, "TRUE =", TRUE)

(500, 4) SENS = 1794.0 TRUE = 899000


In [16]:
# 2) 파이프라인 조립 (num_fix_attempts=2 → 에러 수정 LLM 호출 정확히 2회, 총 최대 3회)
pipe = W.build_pipe_nosr(model="gpt-4o-mini", temperature=0.0, num_fix_attempts=2)

Loading annotations from semeval train[:400]


In [22]:
# 3) 프롬프트 검증 — 예시 9개까지 포함한 최종 문자열에 SR 이 없어야 한다
row = {"question": QUESTION, "dataset": DS_NAME}
p = W.assert_no_sample_rows(pipe, row)
print("prompt chars:", len(p))
print(p[-600:])                                 # 꼬리: 마지막 CUAT 라벨에서 끊겨 있어야 함

prompt chars: 38145
hest philanthropy score)
    return top_philanthropist['gender']
    
# TODO: complete the following function. It should give the answer to: What is the highest Credit_Limit?
def answer(df: pd.DataFrame):
    """
        #,Column,Non-Null CounT,Dtype,Types of Elements,Values
        0,Card_Number,500,str,[<class 'str'>],
        1,Card_Family,500,str,[<class 'str'>],
        2,Credit_Limit,500,int64,[<class 'int'>],
        3,Cust_ID,500,str,[<class 'str'>],
    """


    df.columns = ['Card_Number', 'Card_Family', 'Credit_Limit', 'Cust_ID']
    
    # The columns used to answer the question: 


In [23]:
# 4) 1회 실행 + 트레이스
t = pipe.run_one_traced(row)
print(t["output"], "| calls:", t["n_llm_calls"], "| fixes:", t["n_fix_attempts"])
print(t["code"])

899000 | calls: 1 | fixes: 0
    # The columns used to answer the question: # ['Credit_Limit']
    # The types of the columns used to answer the question: ['number[int64]']
    # The type of the answer: number
    
    # Find the maximum value in the 'Credit_Limit' column
    return df['Credit_Limit'].max()
